In [19]:
#imports
import pandas as pd
import numpy as np
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

## Read in adult census data and perform data preprocessing

In [26]:
# importing the data
adult = pd.read_csv('/Users/donyabehroozi/Documents/gsb545/GSB-545/In Class Assignments/adult.csv')
adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [27]:
#data preprocessing

# replace ? with np.nan
adult = adult.replace("?", np.nan)
adult.head()

# convert target variable to binary
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

# convert gender to 0/1 (doesn't need categorical encoding since it's binary)
if "gender" in adult.columns:
    adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)

# drop the fnlwgt variable as it is not useful for modeling
adult.drop(columns=["fnlwgt"], inplace=True)  
    
adult.head(20)

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0
5,34,Private,10th,6,Never-married,Other-service,Not-in-family,White,1,0,0,30,United-States,0
6,29,NaN,HS-grad,9,Never-married,NaN,Unmarried,Black,1,0,0,40,United-States,0
7,63,Self-emp-not-inc,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,1,3103,0,32,United-States,1
8,24,Private,Some-college,10,Never-married,Other-service,Unmarried,White,0,0,0,40,United-States,0
9,55,Private,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,1,0,0,10,United-States,0


In [28]:
#feature engineering

#net capital activity
adult['capital_net'] = adult['capital-gain'] - adult['capital-loss'] 

#age and education interaction
adult['age_education_interaction'] = adult['age'] * adult['educational-num']

#log transformation of net capital activity to help models handle extreme values
capital_net_clipped = adult['capital_net'].clip(lower=0)
adult['capital_net_log'] = np.log1p(capital_net_clipped)

#age and marriage interaction
adult['age_married'] = adult['age'] * (adult['marital-status'] == 'Married-civ-spouse').astype(int)

adult.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,capital_net,age_education_interaction,capital_net_log,age_married
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,0,175,0.000000,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,0,342,0.000000,38
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1,0,336,0.000000,28
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1,7688,440,8.947546,44
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0,0,180,0.000000,0


In [29]:
#count NAs
adult.isna().sum()

age                             0
workclass                    2799
education                       0
educational-num                 0
marital-status                  0
occupation                   2809
relationship                    0
race                            0
gender                          0
capital-gain                    0
capital-loss                    0
hours-per-week                  0
native-country                857
income                          0
capital_net                     0
age_education_interaction       0
capital_net_log                 0
age_married                     0
dtype: int64

In [30]:
#impute missing values with "unknown"
for col in ['workclass', 'occupation', 'native-country']:
    adult[col] = adult[col].fillna('Unknown')

adult.head(20)

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,capital_net,age_education_interaction,capital_net_log,age_married
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,0,175,0.000000,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,0,342,0.000000,38
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1,0,336,0.000000,28
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1,7688,440,8.947546,44
4,18,Unknown,Some-college,10,Never-married,Unknown,Own-child,White,0,0,0,30,United-States,0,0,180,0.000000,0
5,34,Private,10th,6,Never-married,Other-service,Not-in-family,White,1,0,0,30,United-States,0,0,204,0.000000,0
6,29,Unknown,HS-grad,9,Never-married,Unknown,Unmarried,Black,1,0,0,40,United-States,0,0,261,0.000000,0
7,63,Self-emp-not-inc,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,1,3103,0,32,United-States,1,3103,945,8.040447,63
8,24,Private,Some-college,10,Never-married,Other-service,Unmarried,White,0,0,0,40,United-States,0,0,240,0.000000,0
9,55,Private,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,1,0,0,10,United-States,0,0,220,0.000000,55


In [31]:
# defining X
num_cols = ['age', 'educational-num', 'capital-gain', 'capital-loss',
            'hours-per-week', 'capital_net', 'capital_net_log', 
            'age_education_interaction', 'age_married']

cat_cols = ['workclass', 'education', 'marital-status',
            'occupation', 'relationship', 'race', 'native-country']

X = pd.concat([
    adult[num_cols],
    pd.get_dummies(adult[cat_cols], drop_first=True).astype(int)
], axis=1)

# defining y
y = adult['income'].to_numpy() 

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=321)

# scale numeric columns only
min_max_scaler = MinMaxScaler()
X_train[num_cols] = min_max_scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = min_max_scaler.transform(X_test[num_cols])

print(X_train.head())
print(y_train[:5])

            age  educational-num  capital-gain  capital-loss  hours-per-week  \
48353  0.027397         0.533333           0.0      0.000000        0.397959   
8310   0.301370         0.666667           0.0      0.000000        0.500000   
6063   0.287671         0.733333           0.0      0.000000        0.377551   
8229   0.068493         0.333333           0.0      0.000000        0.397959   
20762  0.109589         0.800000           0.0      0.399679        0.397959   

       capital_net  capital_net_log  age_education_interaction  age_married  \
48353     0.041742              0.0                   0.114200          0.0   
8310      0.041742              0.0                   0.308039          0.0   
6063      0.041742              0.0                   0.328325          0.0   
8229      0.041742              0.0                   0.084899          0.0   
20762     0.025059              0.0                   0.229902          0.0   

       workclass_Local-gov  ...  native-coun

## Baseline neural network

In [33]:
#set random seed
tf.random.set_seed(321)

In [34]:
#construct the model
inputs = keras.Input(shape=(102,)) #102 inputs, input layer
x = layers.Dense(64, activation='relu')(inputs) #hidden layer 1
x = layers.Dense(32, activation='relu')(x) #hidden layer 2
outputs = layers.Dense(1, activation='sigmoid')(x) #output layer (sigmoid for binary outcome)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_model")

In [35]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 443us/step - auc: 0.8832 - loss: 0.3516 - val_auc: 0.8960 - val_loss: 0.3339
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 350us/step - auc: 0.9021 - loss: 0.3253 - val_auc: 0.9001 - val_loss: 0.3275
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 352us/step - auc: 0.9067 - loss: 0.3180 - val_auc: 0.9026 - val_loss: 0.3235
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 354us/step - auc: 0.9102 - loss: 0.3124 - val_auc: 0.9049 - val_loss: 0.3206
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 350us/step - auc: 0.9131 - loss: 0.3077 - val_auc: 0.9065 - val_loss: 0.3183
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 374us/step - auc: 0.9155 - loss: 0.3036 - val_auc: 0.9076 - val_loss: 0.3167
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 349us/step - auc: 0.9175 - loss: 0.3002 - val_auc: 0.9080 - val_loss: 0.3165
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 350us/step - auc: 0.9193 - loss: 0.2971 - val_auc: 0.9078 - val_loss: 0.3169
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 